# Healthy‐Trained 3D Autoencoder with Early Stopping and Anomaly Inference

In [ ]:
#Prepare Colab for work

from google.colab import drive
drive.mount('/content/drive')

#copy dataset to /content for faster access  on colab!
!mkdir /content/data
!cp -r "/content/drive/MyDrive/00-DataScience_BIU/Final Project/data/IXI-T1_resampled" /content/data/IXI-T1_resampled/
!cp -r "/content/drive/MyDrive/00-DataScience_BIU/Final Project/data/T1_tumor_resampled" /content/data/T1_tumor_resampled/
!cp -r "/content/drive/MyDrive/00-DataScience_BIU/Final Project/data/T1Gd_tumor_resampled" /content/data/T1Gd_tumor_resampled/

!pip install SimpleITK scikit-learn #piqa

Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/00-DataScience_BIU/Final Project/data/T1Gd_tumor_resampled': No such file or directory
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 17.8 MB/s eta 0:00:00


In [ ]:
!nvidia-smi

Wed Jun  4 19:29:58 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:

import os
import numpy as np
import SimpleITK as sitk
from glob import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
#from piqa import SSIM
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt


In [ ]:

class SmallAE3D(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: 1 → 8 → 16 channels
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 8, 3, padding=1),
            nn.BatchNorm3d(8), nn.ReLU(inplace=True),
            nn.Dropout3d(0.2),
            nn.MaxPool3d(2),                     # → [8, D/2, H/2, W/2]

            nn.Conv3d(8, 16, 3, padding=1),
            nn.BatchNorm3d(16), nn.ReLU(inplace=True),
            nn.Dropout3d(0.2),
            nn.MaxPool3d(2)                      # → [16, D/4, H/4, W/4]
        )
        # Decoder: 16 → 8 → 1 channels
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(16, 8, 2, stride=2),
            nn.BatchNorm3d(8), nn.ReLU(inplace=True),  # → [8, D/2, H/2, W/2]

            nn.ConvTranspose3d(8, 1, 2, stride=2),  # → [1, D, H, W]
            # Linear output
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


In [ ]:

class HealthyNiftiDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = sitk.ReadImage(self.paths[idx])
        arr = sitk.GetArrayFromImage(img).astype(np.float32)  # [D, H, W]
        # Z-score → min-max normalize to [0,1]
        mean, std = arr.mean(), arr.std()
        norm = (arr - mean) / (std + 1e-5)
        norm = (norm - norm.min()) / (norm.max() - norm.min() + 1e-5)
        norm = np.clip(norm, 0.0, 1.0)
        return torch.tensor(norm[None, ...], dtype=torch.float32)


In [ ]:

# Paths to healthy volumes (update these)
#healthy_paths = sorted(glob("IXI-T1_resampled/*.nii.gz"))
t1_files = sorted(glob(r'/content/data/IXI-T1_resampled/*.nii.gz'))
#t2_files = sorted(glob("/mnt/data/IXI-T2_resampled/*.nii.gz"))
healthy_paths = t1_files # + t2_files
#healthy_paths = sorted(glob("IXI-T1_resampled/*.nii.gz"))
# Split into train/val
train_paths, val_paths = train_test_split(healthy_paths, test_size=0.2, random_state=42)

train_ds = HealthyNiftiDataset(train_paths)
val_ds   = HealthyNiftiDataset(val_paths)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=1)


In [ ]:

# Model, optimizer, loss, SSIM
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallAE3D().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-6)
mse_loss = nn.MSELoss()
ssim_loss = SSIM(n_channels=1).to(device)
writer = SummaryWriter(log_dir=f"/content/drive/MyDrive/00-DataScience_BIU/Final Project/runs/AE_Healthy_T1_1ch_ep")


NameError: name 'SSIM' is not defined

In [ ]:

# Training with early stopping
patience = 5
best_val_loss = float("inf")
epochs_without_improvement = 0
n_epochs = 50
train_losses, val_losses = [], []

for epoch in range(1, n_epochs + 1):
    model.train()
    train_epoch_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        recon = model(batch)
        recon_clamped = torch.clamp(recon, 0.0, 1.0)
        loss = 0.8 * mse_loss(recon, batch) + 0.2 * (1 - ssim_loss(recon_clamped, batch))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_epoch_loss += loss.item()
    train_epoch_loss /= len(train_loader)
    train_losses.append(train_epoch_loss)

    model.eval()
    val_epoch_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            recon = model(batch)
            recon_clamped = torch.clamp(recon, 0.0, 1.0)
            loss = 0.8 * mse_loss(recon, batch) + 0.2 * (1 - ssim_loss(recon_clamped, batch))
            val_epoch_loss += loss.item()
    val_epoch_loss /= len(val_loader)
    val_losses.append(val_epoch_loss)

    writer.add_scalars("Loss", {"Train": train_epoch_loss, "Val": val_epoch_loss}, epoch)
    print(f"Epoch {epoch:02d} | Train Loss: {train_epoch_loss:.4f} | Val Loss: {val_epoch_loss:.4f}")

    if val_epoch_loss < best_val_loss:
        best_val_loss = val_epoch_loss
        epochs_without_improvement = 0
        model_path = f"/content/drive/MyDrive/00-DataScience_BIU/Final Project/AE_Healthy_T1_1ch_ep({epoch})_loss({val_epoch_loss:.4f}).pt"
        torch.save(model.state_dict(), model_path) #"small_ae_healthy_best.pt")
        print("  ✔️ Validation improved; model saved.")
    else:
        epochs_without_improvement += 1
        print(f"  ❌ No improvement for {epochs_without_improvement}/{patience} epochs.")
        if epochs_without_improvement >= patience:
            print(f"Stopping early at epoch {epoch}.")
            break


Epoch 01 | Train Loss: 0.2541 | Val Loss: 0.1827
  ✔️ Validation improved; model saved.
Epoch 02 | Train Loss: 0.1617 | Val Loss: 0.1282
  ✔️ Validation improved; model saved.
Epoch 03 | Train Loss: 0.1245 | Val Loss: 0.1050
  ✔️ Validation improved; model saved.
Epoch 04 | Train Loss: 0.1079 | Val Loss: 0.0932
  ✔️ Validation improved; model saved.
Epoch 05 | Train Loss: 0.0982 | Val Loss: 0.0854
  ✔️ Validation improved; model saved.
Epoch 06 | Train Loss: 0.0910 | Val Loss: 0.0789
  ✔️ Validation improved; model saved.
Epoch 07 | Train Loss: 0.0847 | Val Loss: 0.0724
  ✔️ Validation improved; model saved.
Epoch 08 | Train Loss: 0.0779 | Val Loss: 0.0629
  ✔️ Validation improved; model saved.
Epoch 09 | Train Loss: 0.0708 | Val Loss: 0.0560
  ✔️ Validation improved; model saved.
Epoch 10 | Train Loss: 0.0653 | Val Loss: 0.0525
  ✔️ Validation improved; model saved.
Epoch 11 | Train Loss: 0.0627 | Val Loss: 0.0545
  ❌ No improvement for 1/5 epochs.
Epoch 12 | Train Loss: 0.0607 | Val 

In [ ]:

# Plot training and validation loss curves
plt.figure(figsize=(8,5))
plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss")
plt.plot(range(1, len(val_losses)+1), val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(r"/content/drive/MyDrive/00-DataScience_BIU/Final Project/AE_Healthy_T1_1ch.png")
plt.show()


NameError: name 'train_losses' is not defined

<Figure size 800x500 with 0 Axes>

# Inference starts here

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallAE3D().to(device)

In [ ]:
#Paths

epoch_num = 46
model_path = f"/content/drive/MyDrive/00-DataScience_BIU/Final Project/AE_Healthy/AE_Healthy_T1_1ch_ep(46)_loss(0.0402).pt"
infer_num = '043'
scan_type = 'T1' #'T1Gd'
input_img = f"BRATS_{infer_num}_{scan_type}.nii.gz"
tumor_path = f"/content/drive/MyDrive/00-DataScience_BIU/Final Project/data/{scan_type}_tumor_resampled/{input_img}"

tumor_path


'/content/drive/MyDrive/00-DataScience_BIU/Final Project/data/T1_tumor_resampled/BRATS_043_T1.nii.gz'

In [ ]:
# Load best model for inference

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallAE3D().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))


<All keys matched successfully>

In [12]:

# Inference on a tumor case (update path)
tumor_img = sitk.ReadImage(tumor_path)
tumor_arr = sitk.GetArrayFromImage(tumor_img).astype(np.float32)
mean, std = tumor_arr.mean(), tumor_arr.std()
norm = (tumor_arr - mean) / (std + 1e-5)
norm = (norm - norm.min()) / (norm.max() - norm.min() + 1e-5)
norm = np.clip(norm, 0.0, 1.0)

with torch.no_grad():
    inp = torch.tensor(norm[None, None, ...], dtype=torch.float32).to(device)
    out = model(inp).cpu().numpy()[0, 0]

# Denormalize and compute error map
recon = out * (std + 1e-5) + mean
err_map = np.abs(tumor_arr - recon)

# Save reconstructed volume
recon_itk = sitk.GetImageFromArray(recon.astype(np.float32))
recon_itk.SetOrigin(tumor_img.GetOrigin())
recon_itk.SetSpacing(tumor_img.GetSpacing())
recon_itk.SetDirection(tumor_img.GetDirection())
sitk.WriteImage(recon_itk, f"/content/drive/MyDrive/00-DataScience_BIU/Final Project/AE_Healthy/AE_{infer_num}_{scan_type}_ep{epoch_num}_recon.nii.gz")


# Save error map
err_itk = sitk.GetImageFromArray(err_map.astype(np.float32))
err_itk.SetOrigin(tumor_img.GetOrigin())
err_itk.SetSpacing(tumor_img.GetSpacing())
err_itk.SetDirection(tumor_img.GetDirection())
sitk.WriteImage(err_itk, f"/content/drive/MyDrive/00-DataScience_BIU/Final Project/AE_Healthy/AE_{infer_num}_{scan_type}_ep{epoch_num}_tumor_error_map.nii.gz")

# Threshold at 99.5th percentile
#thresh = np.percentile(err_map, 99.5)
#mask = (err_map >= thresh).astype(np.uint8)

#---------------------------
# Define custom high-end percentiles
p1 = np.percentile(err_map, 99.2)   # High error
p2 = np.percentile(err_map, 99.5)   # Very high error
p3 = np.percentile(err_map, 99.8)  # Extreme error

# Initialize mask as background
mask = np.zeros_like(err_map, dtype=np.uint8)

# Assign class labels
mask[(err_map >= p1) & (err_map < p2)] = 3   # High error Red=1
mask[(err_map >= p2) & (err_map < p3)] = 2   # Very high error Green=2
mask[err_map >= p3] = 1                      # Extreme error Blue=3
#---------------------------


# Save binary anomaly mask
mask_itk = sitk.GetImageFromArray(mask)
mask_itk.SetOrigin(tumor_img.GetOrigin())
mask_itk.SetSpacing(tumor_img.GetSpacing())
mask_itk.SetDirection(tumor_img.GetDirection())
sitk.WriteImage(mask_itk, f"/content/drive/MyDrive/00-DataScience_BIU/Final Project/AE_Healthy/AE_{infer_num}_{scan_type}_ep{epoch_num}_tumor_anomaly_mask.nii.gz")

#print(f"Threshold used: {thresh:.4f}")


In [ ]:

# Visualize center slice of original, reconstruction, error, and mask
slice_idx = 154 # tumor_arr.shape[0] // 2

plt.figure(figsize=(16,16))

plt.subplot(2,2,1)
plt.imshow(tumor_arr[slice_idx], cmap='gray')
plt.title("Original Tumor")
plt.axis('off')

plt.subplot(2,2,2)
plt.imshow(recon[slice_idx], cmap='gray')
plt.title("Reconstruction")
plt.axis('off')

plt.subplot(2,2,3)
plt.imshow(err_map[slice_idx], cmap='hot')
plt.title("Error Map")
plt.axis('off')

plt.subplot(2,2,4)
plt.imshow(tumor_arr[slice_idx], cmap='gray', alpha=0.5)
plt.imshow(mask[slice_idx], cmap='Reds', alpha=0.5)
plt.title("Anomaly Mask Overlay")
plt.axis('off')

plt.suptitle("Inference Results")
plt.savefig("tumor_inference_visual.png")
plt.show()
